## Part 1  Generate Datasets

In [ ]:
#!pip install faker

In [ ]:
import random
from faker import Faker 
import string
import time
from datetime import date
from random import randrange
from datetime import timedelta
from datetime import datetime
import pandas as pd

In [ ]:
fake = Faker() 

In [ ]:
# fake ID generator

# number only, used for user_id etc
def generate_id_num(length=10):
    characters = string.digits #string.ascii_letters
    return ''.join(random.choice(characters) for i in range(length))

# with certain prefix, used for games etc
def generate_id_prefix(prefix, length=8):
    characters = string.digits #string.ascii_letters
    return prefix+''.join(random.choice(characters) for i in range(length))

# time-based, for transactional record, e.g.bets
def datetime_to_float(d):
    epoch = datetime.utcfromtimestamp(0)
    total_seconds =  (d - epoch).total_seconds()
    return total_seconds

def generate_id_time(prefix, timestamp,length=6):
    ts = int(datetime_to_float(timestamp))
    if length == 0:
        return f"{prefix}-{ts}"
    else:
        characters = string.digits+string.ascii_letters
        sequence = ''.join(random.choice(characters) for i in range(length))
        return f"{prefix}-{ts}-{sequence}"

In [ ]:
# Timestamp generator
def random_date(start, end):
    delta = end - start
    int_delta = (delta.days * 24 * 60 * 60) + delta.seconds
    random_second = randrange(int_delta)
    return start + timedelta(seconds=random_second)

In [ ]:
# create fake user profile
def create_user(n):
    users = []
    user_ids = []
    while len(users) < n:
        # user elements: id, name, dob, age, res_address, res_country, email, created_at, updated_at
        user_id = ' '
        while user_id not in user_ids:
            user_id = generate_id_num()
            if user_id in user_ids:
                continue
            else:   
                name = fake.name()
                gender = random.choices(['Female','Male', 'None of Above'], weights = [30,70,2])
                dob = fake.date_of_birth()
                today = date.today()
                age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
                if age < 18 or age > 75:
                    continue
                else:
                    address = fake.address().replace('\n',' ')
                    country = fake.country()
                    email = fake.email()
                    start_date = datetime.strptime('2022-01-01', '%Y-%m-%d')
                    current = datetime.today()
                    created_at = random_date(start_date, current)
                    updated_at = random_date(created_at, current)

                    user = {'id': user_id,
                            'name': name,
                            'dob': dob.strftime("%Y-%m-%d"),
                            'age': age,
                            'gender': gender[0],
                            'address': address,
                            'country': country,
                            'email': name.lower().replace(' ','').replace('.','')+email[email.index('@'):],
                            'created_at': created_at.strftime("%Y-%m-%d %H:%M:%S"),
                            'updated_at': updated_at.strftime("%Y-%m-%d %H:%M:%S")}
                    user_ids.append(user_id)
                    users.append(user)
    return users

In [ ]:
users = create_user(500)
users_df = pd.DataFrame(users)

In [ ]:
users_df.head(5)

In [ ]:
users_df.to_csv('users.csv', sep = "|", index = False)

In [ ]:
# create user_kyc data
def kyc_gen(users_df):
    records = []
    for u in users_df['id'].unique():
        
        n = random.choices([1,2,3],weights = [15,4,1])[0] # number of kyc records for this profile as kyc can be re-attampted
        kyc_level = random.choices(['Standard', 'Simplified'], weights=[8,3])[0]
        start_dt = users_df.loc[users_df.id==u, 'created_at'].values[0]
        start = datetime.strptime(start_dt, "%Y-%m-%d %H:%M:%S")
        end = datetime.today()
        user_kyc = []
        for i in range(1,n+1):
            if i < n:
                kyc_created_at = random_date(start, end)
                start = kyc_created_at
                outcome = random.choices(['Unconfirmed', 'Cancelled'], weights = [8,2])[0]
                kyc = {'user_id':u,
                       'kyc_level': kyc_level,
                       'kyc_outcome': outcome,
                       'created_at': kyc_created_at}
                user_kyc.append(kyc)
            if i==n:
                kyc_created_at = random_date(start, end)
                kyc = {'user_id':u,
                       'kyc_level': kyc_level,
                       'kyc_outcome': 'Confirmed',
                       'created_at': kyc_created_at.strftime("%Y-%m-%d %H:%M:%S")}
                user_kyc.append(kyc)
        records = records+user_kyc
    return records

In [ ]:
kyc_records = kyc_gen(users_df)
kyc_df = pd.DataFrame(kyc_records)

In [ ]:
kyc_df.head(5)

In [ ]:
len(kyc_df)

In [ ]:
kyc_df.to_csv('user_kyc.csv', sep = "|", index = False)

In [ ]:
# creat wallet data
def wallet_gen(users_df):
    records = []
    for u in users_df['id'].unique():
        
        n = random.choices([1,2,3,4],weights = [50,38,8,1])[0] # number of wallets for this profile
        start_dt = users_df.loc[users_df.id==u, 'created_at'].values[0]
        start = datetime.strptime(start_dt, "%Y-%m-%d %H:%M:%S")
        end = datetime.today()
        currs = []
        wallets = []
        while len(currs) < n:
            curr = fake.currency()[0]
            if curr in currs:
                continue
            else:
                wallet_id = 'wa'+u+'-'+curr.lower()
                balance = round(random.uniform(15, 1000),2)
                wallet_created_at = random_date(start, end)
                wallet_updated_at = random_date(wallet_created_at, end)
                wallet = {
                        'id': wallet_id,
                        'user_id':u,
                        'currency': curr,
                        'current_balance': balance,
                        'created_at': wallet_created_at.strftime("%Y-%m-%d %H:%M:%S"),
                        'upadted_at': wallet_updated_at.strftime("%Y-%m-%d %H:%M:%S")}
                currs.append(curr)
                wallets.append(wallet)
        records = records+wallets
    return records

In [ ]:
wallet_records = wallet_gen(users_df)
wallet_df = pd.DataFrame(wallet_records)

In [ ]:
wallet_df.head(5)

In [ ]:
len(wallet_df)

In [ ]:
wallet_df.to_csv('user_wallet.csv', sep = "|", index = False)

In [ ]:
# create deposit data
def deposit_gen(wallet_df):
    records = []
    for w in wallet_df['id'].unique():
        
        n = round(random.uniform(10, 50)) # number of deposits for each wallet
        start_dt = wallet_df.loc[wallet_df.id==w, 'created_at'].values[0]
        start = datetime.strptime(start_dt, "%Y-%m-%d %H:%M:%S")
        end = datetime.today()

        deposits = [] 
        
        while len(deposits) < n:
            deposit_created_at = random_date(start, end)    
            deposit_id = generate_id_time(w, deposit_created_at) # timestamp based id, no duplicates
            value = round(random.uniform(1, 200),2)
            status = random.choices(['Successful', 'Unsuccessful', 'Cancelled'], weights = [88,2,3])[0]
            deposit = {
                    'id': deposit_id,
                    'wallet_id':w,
                    'value': value,
                    'created_at': deposit_created_at.strftime("%Y-%m-%d %H:%M:%S"),
                    'status': status}
            deposits.append(deposit)
        records = records+deposits
    return records

In [ ]:
deposit_records = deposit_gen(wallet_df)
deposit_df = pd.DataFrame(deposit_records)

In [ ]:
len(deposit_df)

In [ ]:
deposit_df.head(5)

In [ ]:
deposit_df.to_csv('deposit.csv', sep = "|", index = False)

In [ ]:
# create game data
def game_gen(n, gametypes, game_type_codes):
    games = []
    game_ids = []
    
    while len(games) < n:
        game_type = random.choice(gametypes)
        game_type_cd = game_type_codes[game_type]
        game_id = generate_id_prefix(game_type_cd)
        if game_id in game_ids:
            continue
        else:
            name_source = fake.name()
            name_1 = name_source[name_source.index(' '):]
            name_2 = random.choice(['Cup', 'Day', 'Tour'])
            name = game_type+name_1+"'s "+name_2
            
            p = round(random.uniform(1, 10))
            provider = 'Game Provider '+ str(p)
            
            start_date = datetime.strptime('2024-07-01', '%Y-%m-%d')
            current = datetime.today()
            game_created_at = random_date(start_date, current)
            end_date = game_created_at + timedelta(days = 7)
            game_updated_at = random_date(game_created_at, current)
            
            tdelta = current-game_updated_at
            if tdelta.days <= 7:
                status =random.choices(['scheduled', 'ongoing'])[0]
            else:
                status = random.choices(['finished', 'cancelled'], weights = [90,5])[0]
            
            game = {    'id': game_id,
                        'type': game_type,
                        'name': name,
                        'provider': provider,
                        'status': status,
                        'created_at': game_created_at.strftime("%Y-%m-%d %H:%M:%S"),
                        'updated_at': game_updated_at.strftime("%Y-%m-%d %H:%M:%S")}
            
            game_ids.append(game_id)
            games.append(game)
    return games

In [ ]:
game_types = ['Football', 'Basketball', 'Cricket', 'Horse Racing', 'Greyhounds']
game_type_codes = {'Football': 'FOTY',
                  'Basketball': 'BSKT',
                  'Cricket': 'CRKT',
                  'Horse Racing': 'HRSR',
                  'Greyhounds': 'GRHR'}

In [ ]:
games = game_gen(100, game_types, game_type_codes)
games_df = pd.DataFrame(games)

In [ ]:
len(games_df)

In [ ]:
games_df.head(5)

In [ ]:
games_df.to_csv('game.csv', sep = "|", index = False)

In [ ]:
# creat bets table based on games and users
def bets_gen(users_df, games_df):
    
    records = []
    users = users_df['id'].unique()
    
    for g in games_df['id'].unique():
        n = round(random.uniform(0, 500)) # number of bets for this game
        game_bets = []
        
        while len(game_bets) < n:
            
            start_dt = games_df.loc[games_df.id==g, 'created_at'].values[0]
            start = datetime.strptime(start_dt, "%Y-%m-%d %H:%M:%S")
            end_dt = games_df.loc[games_df.id==g, 'updated_at'].values[0]
            end = datetime.strptime(end_dt, "%Y-%m-%d %H:%M:%S")

            bet_created_at = random_date(start, end)
            bet_id = generate_id_time(g, bet_created_at)
            user_id = random.choice(users)
            value = round(random.uniform(1, 100),2)
            mlp = round(random.uniform(-1.8, 3),2)
            if mlp < 0:
                profit =0
            else:
                profit = round((mlp-1.00)*value,2)
            bet_updated_at = random_date(bet_created_at, end)
            game_status = games_df.loc[games_df.id==g, 'status'].values[0]

            if game_status == 'ongoing':
                bet_status = 'active'
            elif game_status == 'scheduled':
                bet_status = 'pending'
            elif game_status == 'cancelled':
                bet_status = 'cancelled'
            elif game_status == 'finished':
                bet_status = 'settled'
            bet = {
                'id': bet_id,
                'user_id': user_id,
                'game_id': g,
                'value': value,
                'status': bet_status,
                'multiplier': mlp,
                'profit': profit,
                'created_at': bet_created_at.strftime("%Y-%m-%d %H:%M:%S"),
                'updated_at': bet_updated_at.strftime("%Y-%m-%d %H:%M:%S")
            }
            game_bets.append(bet)
    
        records = records + game_bets
    return records
        

In [ ]:
users_df = pd.read_csv('data/users.csv', dtype = {'id': 'object'}, sep = "|")
games_df = pd.read_csv('data/game.csv', dtype = {'id': 'object'}, sep = "|")

In [ ]:
bets = bets_gen(users_df, games_df)
bets_df = pd.DataFrame(bets)

In [ ]:
len(bets_df)

In [ ]:
bets_df[bets_df['status'] == 'active']

In [ ]:
bets_df.head(5)

In [ ]:
bets_df.to_csv('bets.csv', sep = "|", index = False)

In [ ]:
#wallet_df = pd.read_csv('data/wallet.csv', dtype = {'user_id': 'object'}, sep = "|")

In [ ]:
def exchange_rate_gen(wallet_df):
    records = []
    
    for c in wallet_df['currency'].unique():
        
        n = round(random.uniform(1, 8)) # number of exchange rate records
        start = datetime.strptime('2025-01-01', '%Y-%m-%d')
        end = datetime.today()
        rates = [] 
        rate_ids = []
        while len(rates) < n:
            rate_created_at = random_date(start, end)
            start = rate_created_at  
            rate_id = generate_id_time('EX'+c, rate_created_at, length = 0)

            if rate_id in rate_ids:
                continue
            else:
                cur_rate = round(random.uniform(0, 1.2), 4)
                rate = {
                    'id': rate_id,
                    'currency': c,
                    'rate': cur_rate,
                    'created_at': rate_created_at.strftime("%Y-%m-%d %H:%M:%S")
                }
            
                rates.append(rate)
                rate_ids.append(rate_id)
        records = records+rates
    return records

In [ ]:
rates = exchange_rate_gen(wallet_df)
rates_df = pd.DataFrame(rates)

In [ ]:
rates_df.head(5)

In [ ]:
rates_df.to_csv("rates.csv", sep = "|", index = False)